Este notebook irá montar uma tabela com todos os dados dos Municipios Brasileiros extraidos da API disponibilizada

pelo  IPEA (Instituto de Pesquisa Econômica Aplicada)

<pre>
Repositório oficial:
    https://github.com/ipea/geobr

Documentação:
    https://ipea.github.io/geobr/

</pre>

In [1]:
import os, sys, geobr
from pyspark.sql import functions as F

In [2]:
# Adiciona a pasta raiz do projeto (um ou dois níveis acima) no caminho do Python
sys.path.append(os.path.abspath(os.path.join('..')))  # Ajuste a quantidade de '..' conforme a profundidade da subpasta

# Cria uma conexão Spark 
from spark_utils import get_spark_session # ver em C:\Marco Conti\Projetos\MAIS-v2\spark_utils.py
spark = get_spark_session("IPEA_municipios")


In [3]:
gdf_ibge_simplified_False = \
    geobr.read_municipality(code_muni="all"
                           ,year=2022
                           ,simplified=False)

gdf_ibge_simplified_True = \
    geobr.read_municipality(code_muni="all"
                           ,year=2022
                           ,simplified=True)

c:\Marco Conti\Projetos\mais_einstein\.venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'github.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Marco Conti\Projetos\mais_einstein\.venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'release-assets.githubusercontent.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
municipalities_2022.parquet: 100%|██████████| 269M/269M [00:51<00:00, 5.20MB/s] 


In [5]:
# converte geometria para texto WKT
gdf_ibge_simplified_False["geometry_wkt"] = gdf_ibge_simplified_False.geometry.to_wkt()

# remove coluna geometry do geopandas
pdf = gdf_ibge_simplified_False.drop(columns=["geometry"])

# cria Spark DataFrame
df_spark = spark.createDataFrame(pdf)

In [7]:
df_spark.printSchema()
df_spark.count()

root
 |-- code_muni: double (nullable = true)
 |-- name_muni: string (nullable = true)
 |-- code_state: double (nullable = true)
 |-- abbrev_state: string (nullable = true)
 |-- name_state: string (nullable = true)
 |-- code_region: double (nullable = true)
 |-- name_region: string (nullable = true)
 |-- year: double (nullable = true)
 |-- geometry_wkt: string (nullable = true)



5570

In [8]:
# converte geometria para texto WKT
gdf_ibge_simplified_True["geometry_wkt"] = gdf_ibge_simplified_False.geometry.to_wkt()

# remove coluna geometry do geopandas
pdf_True = gdf_ibge_simplified_False.drop(columns=["geometry"])

# cria Spark DataFrame
df_spark_True = spark.createDataFrame(pdf)

In [9]:
df_spark_True.printSchema()
df_spark_True.count()

root
 |-- code_muni: double (nullable = true)
 |-- name_muni: string (nullable = true)
 |-- code_state: double (nullable = true)
 |-- abbrev_state: string (nullable = true)
 |-- name_state: string (nullable = true)
 |-- code_region: double (nullable = true)
 |-- name_region: string (nullable = true)
 |-- year: double (nullable = true)
 |-- geometry_wkt: string (nullable = true)



5570

In [10]:
df_spark.createOrReplaceTempView("temp_false")
df_spark_True.createOrReplaceTempView("temp_true")

In [13]:
query = \
    """with f as (Select 'false' as origem, * from temp_false limit 10)
           ,t as (Select 'true' as origem, * from temp_true limit 10)
       Select * from f
       Union all
       Select * from t 
       order by code_muni
    """

spark.sql(query).show(100,False)

+------+---------+---------------------+----------+------------+----------+-----------+-----------+------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------